# API Konzept

Get: WAPapi/v1/route?

- start_point = [lat, long | Adresse]
- end_point = [lat, long | Adresse]
- start_time = int [Timestamp]
- speed = float [km/h]
- routingmodel = str ['einfach'|'funktion'|'rerouting'|'all']
- sensibility = str ['low'|'medium'|'high']


# API RND (Research and Development)

In [6]:
# Beispielabfrage

start_point = 'Johann Brüderlin-Strasse 15, 4132 Muttenz'
end_point = 'Hollenweg 8A, 4153 Reinach'
start_time = 1713650000 + 3600 *5
speed = 10
routingmodel = 'advanced'
sensibility = 'low' 


In [8]:
import osmnx as ox
import xarray as xr
import networkx as nx 
from pathlib import Path
import numpy as np
import heapq




In [ ]:
#from get_nc_file import get_nc_file
#from get_nearest_node import get_nearest_node
from utils_graph import _parse_point, get_square_bbox_from_points, get_graph_cached
from utils_nc_file import get_nc_file
from utils_forecast import get_forecast, compute_rain_adjusted_cost
from utils_routingmodels import time_dependent_dijkstra

In [ ]:
def get_route(start_point, end_point, start_time, speed, routingmodel, sensibility):

    # ——————————————————————————————————————————————————————————————————————————
    # Sicherstellen, dass Start-/ Endpunkt im format lat, lon vorliegen
    # ——————————————————————————————————————————————————————————————————————————
    start_point = _parse_point(start_point)
    end_point = _parse_point(end_point)
    
    # ——————————————————————————————————————————————————————————————————————————
    # Speed von km/h in m/s
    # ——————————————————————————————————————————————————————————————————————————
    speed = speed / 3.6

    # ——————————————————————————————————————————————————————————————————————————
    # Aus Start-/ Endpunkt den richtigen Graphen aus dem Cache finden oder herunterladen
    # ——————————————————————————————————————————————————————————————————————————
    bbox = get_square_bbox_from_points(start_point, end_point)
    G = get_graph_cached(bbox)
    

    # ——————————————————————————————————————————————————————————————————————————
    # Aus Start-/ Endpunkt die richtige Node auswählen
    # ——————————————————————————————————————————————————————————————————————————
    lat_s, lon_s = start_point
    start_node = ox.distance.nearest_nodes(G, lon_s, lat_s)

    lat_e, lon_e = end_point
    end_node = ox.distance.nearest_nodes(G, lon_e, lat_e)


    # ——————————————————————————————————————————————————————————————————————————
    # Richtiges NC-File laden
    # ——————————————————————————————————————————————————————————————————————————
    nc_filepath = str(get_nc_file(start_time))
    ds = xr.open_dataset(nc_filepath)

    # Zeitstempel als integer vorbereiten
    start_time = int(start_time)
    nc_file_timestamp = int(Path(nc_filepath).stem)


    # ——————————————————————————————————————————————————————————————————————————
    # Routing anhand gewähltem Routingmodel durchführen
    # ——————————————————————————————————————————————————————————————————————————
    if routingmodel == 'einfach':

        for edge in G.edges(keys=True, data=True):
            u, v, k, data = edge
            data["forecast"] = get_forecast(G, ds, u, v, k, 
                                        file_timestamp=nc_file_timestamp, 
                                        target_timestamp=start_time,
                                        interpolate=False)
            
            data['cost'] = compute_rain_adjusted_cost(data['length'], data["forecast"], sensibility)

            data['travel_time'] = int(data['length'] / speed)

        route = ox.routing.shortest_path(G, start_node, end_node, weight='cost')
    
    # TODO
    elif routingmodel == 'advanced':
    
        route = time_dependent_dijkstra(G=G,
                                        start_node=start_node,
                                        end_node=end_node,
                                        start_timestamp=start_time,
                                        speed=speed,
                                        ds=ds,
                                        nc_file_timestamp=nc_file_timestamp,
                                        sensibility=sensibility
    )



    # ——————————————————————————————————————————————————————————————————————————
    # NC-File schliessen
    # ——————————————————————————————————————————————————————————————————————————
    ds.close()


    # ——————————————————————————————————————————————————————————————————————————
    # Ausgabe der Route als geojson
    # ——————————————————————————————————————————————————————————————————————————
    
    route_gdf = ox.routing.route_to_gdf(G, route, weight='cost')
    keep_cols = ["osmid", "length", "cost","travel_time", "geometry"]
    route_gdf = route_gdf[keep_cols]
    return route_gdf.to_json()

    # für debugging -> return G, route



In [ ]:
import heapq

def time_dependent_dijkstra(start_node, end_node, start_timestamp, edges, neighbors_fn):
    """
    start_timestamp: Unix-Timestamp (seconds since 1970)
    """
    dist = {}
    dist[(start_node, start_timestamp)] = 0
    pq = [(0, start_node, start_timestamp)]
    
    while pq:
        cost, loc, current_timestamp = heapq.heappop(pq)
        
        if (loc, current_timestamp) in dist and cost > dist[(loc, current_timestamp)]:
            continue
        
        if loc == goal_loc:
            return cost, current_timestamp  # Ankunftszeit
        
        for neighbor in neighbors_fn(loc):
            travel_time = get_travel_time(loc, neighbor)  # in Sekunden
            arrival_timestamp = current_timestamp + travel_time
            
            edge_cost = edge_cost((loc, neighbor), arrival_timestamp)
            new_cost = cost + edge_cost
            
            if (neighbor, arrival_timestamp) not in dist or new_cost < dist[(neighbor, arrival_timestamp)]:
                dist[(neighbor, arrival_timestamp)] = new_cost
                heapq.heappush(pq, (new_cost, neighbor, arrival_timestamp))
    
    return float('inf'), None

In [10]:
#G, route = get_route(start_point, end_point, start_time, speed, routingmodel, sensibility)
route = get_route(start_point, end_point, start_time, speed, routingmodel, sensibility)


print(route)
#fig, ax = ox.plot.plot_graph_route(G, route, route_color="y", route_linewidth=6, node_size=0)

NameError: name 'heapq' is not defined

#### API gibt ein GeoJSON zurück mit allen Einzellinien => einfach bei mir melden wenn anderes Format gewünscht ist

# Debugging

In [ ]:
fig, ax = ox.plot.plot_graph_route(G, route, route_color="y", route_linewidth=6, route_alpha=0.2, node_size=0)


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

costs = [d["cost"] for _, _, _, d in G.edges(keys=True, data=True)]
norm = plt.Normalize(min(costs), max(costs))
cmap = cm.viridis

route_edges = set(zip(route[:-1], route[1:]))

edge_colors = [
    "red" if (u, v) in route_edges or (v, u) in route_edges
    else mcolors.to_hex(cmap(norm(d["cost"])))
    for u, v, k, d in G.edges(keys=True, data=True)
]

fig, ax = ox.plot_graph(
    G,
    edge_color=edge_colors,
    node_size=0,
    edge_linewidth=1,
    show=False,
    close=False
)

plt.show()

In [ ]:
import numpy as np
import xarray as xr

def diagnose_get_forecast(G, ds, u, v, k, file_timestamp, target_timestamp,
                         eps_idx=0, ref_time_idx=0, var_name="TOT_PREC", interpolate=True):
    """
    Schritt-für-Schritt Diagnose der get_forecast Funktion.
    Zeigt genau wo der Fehler auftritt.
    """
    print("\n" + "="*70)
    print("🔍 DIAGNOSE: get_forecast")
    print("="*70)
    
    # ═══════════════════════════════════════════════
    # 1. Edge + Raster
    # ═══════════════════════════════════════════════
    print("\n[SCHRITT 1] Edge + Raster extrahieren")
    try:
        edge = G.edges[u, v, k]
        print(f"  ✅ Edge gefunden: {u} → {v} (key={k})")
        print(f"     Edge-Daten: {dict(edge)}")
        
        if "cell_i" not in edge or "cell_j" not in edge:
            print(f"  ❌ FEHLER: Edge hat keine cell_i/cell_j!")
            print(f"     Verfügbare Keys: {list(edge.keys())}")
            return None
        
        i = edge["cell_i"]
        j = edge["cell_j"]
        print(f"  ✅ Cell indices: i={i}, j={j}")
    except Exception as e:
        print(f"  ❌ FEHLER beim Edge-Zugriff: {e}")
        return None
    
    # ═══════════════════════════════════════════════
    # 2. Zeit → lead_time
    # ═══════════════════════════════════════════════
    print("\n[SCHRITT 2] Zeit → lead_time")
    try:
        lead_hours = (target_timestamp - file_timestamp) / 3600.0
        print(f"  ✅ file_timestamp: {file_timestamp}")
        print(f"  ✅ target_timestamp: {target_timestamp}")
        print(f"  ✅ lead_hours: {lead_hours}")
        
        if lead_hours < 0:
            print(f"  ❌ FEHLER: target_timestamp liegt vor Modellstart!")
            return None
        
        da = ds[var_name]
        print(f"  ✅ Variable '{var_name}' extrahiert")
        print(f"     Dimensions: {da.dims}")
        print(f"     Shape: {da.shape}")
        
        if "lead_time" not in da.dims:
            print(f"  ❌ FEHLER: lead_time nicht in Dimensions!")
            print(f"     Verfügbare Dims: {da.dims}")
            return None
        
        print(f"  ✅ lead_time Dimension vorhanden")
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════
    # 3. Interpolation ODER Rundung
    # ═══════════════════════════════════════════════
    print("\n[SCHRITT 3] Interpolation / Rundung")
    try:
        if interpolate:
            print(f"  → Lineare Interpolation bei lead_time={lead_hours}")
            da_t = da.interp(lead_time=lead_hours)
            print(f"  ✅ Nach Interpolation: {da_t.dims} | {da_t.shape}")
        else:
            lead_idx = int(np.round(lead_hours))
            print(f"  → Rundung: lead_hours={lead_hours} → lead_idx={lead_idx}")
            print(f"     lead_time Koordinaten: {da['lead_time'].values}")
            print(f"     lead_time min/max: {da['lead_time'].min().item()}, {da['lead_time'].max().item()}")
            
            if lead_idx >= len(da['lead_time']):
                print(f"  ⚠️  WARNUNG: lead_idx={lead_idx} überschreitet Array-Länge!")
                print(f"     Setze auf Maximum: {len(da['lead_time'])-1}")
                lead_idx = len(da['lead_time']) - 1
            
            da_t = da.isel(lead_time=lead_idx)
            print(f"  ✅ Nach isel(lead_time={lead_idx}): {da_t.dims} | {da_t.shape}")
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════
    # 4. Raum + Ensemble + Zeit extrahieren
    # ═══════════════════════════════════════════════
    print("\n[SCHRITT 4] Finale Indexierung")
    print(f"  da_t.dims nach Interpolation: {da_t.dims}")
    print(f"  da_t.shape: {da_t.shape}")
    
    # Prüfe alle Dimensionen
    for dim_name in ['eps', 'ref_time', 'y', 'x']:
        if dim_name in da_t.dims:
            dim_size = da_t.sizes[dim_name]
            print(f"  ✅ '{dim_name}' Dimension: Größe={dim_size}")
        else:
            print(f"  ⚠️  '{dim_name}' Dimension: NICHT VORHANDEN!")
    
    try:
        print(f"\n  Extrahiere mit:")
        print(f"    eps={eps_idx}, ref_time={ref_time_idx}, y={i}, x={j}")
        
        # Validiere Indices
        if eps_idx >= da_t.sizes.get('eps', 1):
            print(f"  ❌ eps_idx={eps_idx} zu groß (max: {da_t.sizes.get('eps', 1)-1})")
            return None
        if ref_time_idx >= da_t.sizes.get('ref_time', 1):
            print(f"  ❌ ref_time_idx={ref_time_idx} zu groß (max: {da_t.sizes.get('ref_time', 1)-1})")
            return None
        if i >= da_t.sizes.get('y', 1):
            print(f"  ❌ i={i} zu groß (max: {da_t.sizes.get('y', 1)-1})")
            return None
        if j >= da_t.sizes.get('x', 1):
            print(f"  ❌ j={j} zu groß (max: {da_t.sizes.get('x', 1)-1})")
            return None
        
        value = da_t.isel(
            eps=eps_idx,
            ref_time=ref_time_idx,
            y=i,
            x=j
        ).item()
        
        print(f"  ✅ Wert erfolgreich extrahiert: {value}")
        return float(value)
    
    except Exception as e:
        print(f"  ❌ FEHLER bei isel: {type(e).__name__}: {e}")
        print(f"\n  Debugging Info:")
        print(f"    da_t.dims: {da_t.dims}")
        print(f"    da_t.shape: {da_t.shape}")
        print(f"    da_t.sizes: {dict(da_t.sizes)}")
        return None
    
    print("\n" + "="*70 + "\n")

In [ ]:
for u, v, key, data in G.edges(keys=True, data=True):
    print(u, v, key)
    print(data)

    break

In [ ]:

# Ersetze diese mit deinen echten Werten:
u, v, k =  21598027, 31075724, 0 # Eine Edge aus G.edges(keys=True)
nc_file_timestamp = int(Path(nc_filepath).stem)
start_time = 1713700000  # dein Zielzeitpunkt

result = diagnose_get_forecast(
    G, ds, u, v, k,
    file_timestamp=nc_file_timestamp,
    target_timestamp=start_time,
    interpolate=False
)

print(f"\n✅ Ergebnis: {result}")

In [ ]:
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree

def diagnose_get_cellid(G, nc_filepath="nc_folder/NC_for_Cellid.nc", lat_name="lat", lon_name="lon"):
    """
    Diagnostiziert Probleme bei der cell_id Vergabe.
    Zeigt genau welche Edges cell_i/cell_j bekommen und welche nicht.
    """
    print("\n" + "="*80)
    print("🔍 DIAGNOSE: get_cellid")
    print("="*80)
    
    # ═══════════════════════════════════════════════════════════════
    # 1. NetCDF laden
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 1] NetCDF laden")
    try:
        ds = xr.open_dataset(nc_filepath)
        print(f"  ✅ Dataset geladen")
        print(f"     Verfügbare Variablen: {list(ds.data_vars)}")
        print(f"     Verfügbare Coordinates: {list(ds.coords)}")
        
        if lat_name not in ds:
            print(f"  ❌ FEHLER: '{lat_name}' nicht im Dataset!")
            print(f"     Verfügbare Optionen: {list(ds.data_vars) + list(ds.coords)}")
            return None
        
        if lon_name not in ds:
            print(f"  ❌ FEHLER: '{lon_name}' nicht im Dataset!")
            print(f"     Verfügbare Optionen: {list(ds.data_vars) + list(ds.coords)}")
            return None
        
        lat = ds[lat_name].values
        lon = ds[lon_name].values
        
        print(f"  ✅ Lat: shape={lat.shape}, range=[{lat.min():.4f}, {lat.max():.4f}]")
        print(f"  ✅ Lon: shape={lon.shape}, range=[{lon.min():.4f}, {lon.max():.4f}]")
        
    except Exception as e:
        print(f"  ❌ FEHLER beim Laden: {e}")
        return None
    
    # ═══════════════════════════════════════════════════════════════
    # 2. KD-Tree erstellen
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 2] KD-Tree erstellen")
    try:
        points = np.column_stack([lon.ravel(), lat.ravel()])
        print(f"  ✅ Grid-Punkte: {len(points)} Punkte")
        print(f"     Shape: {points.shape}")
        
        tree = cKDTree(points)
        print(f"  ✅ KD-Tree erfolgreich erstellt")
        
        # Grid-Indizes
        flat_idx = np.arange(len(points))
        i_all, j_all = np.unravel_index(flat_idx, lat.shape)
        print(f"  ✅ Grid-Indizes erstellt")
        
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════════════════════
    # 3. Edges extrahieren
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 3] Edges extrahieren")
    try:
        all_edges = list(G.edges(keys=True, data=True))
        print(f"  ℹ️  Gesamte Edges: {len(all_edges)}")
        
        # Edges mit Geometrie
        valid_edges = [(u, v, k) for (u, v, k, d) in all_edges if "geometry" in d]
        geoms = [d["geometry"] for (_, _, _, d) in all_edges if "geometry" in d]
        
        edges_without_geom = len(all_edges) - len(valid_edges)
        print(f"  ✅ Edges mit Geometrie: {len(valid_edges)}")
        print(f"  ⚠️  Edges OHNE Geometrie: {edges_without_geom}")
        
        if len(geoms) == 0:
            print(f"  ❌ KRITISCHER FEHLER: Keine Geometrien gefunden!")
            return None
        
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════════════════════
    # 4. Mittelpunkte berechnen
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 4] Mittelpunkte berechnen")
    try:
        midpoints = []
        failed_geoms = []
        
        for idx, geom in enumerate(geoms):
            try:
                midpoint = geom.interpolate(0.5, normalized=True).coords[0]
                midpoints.append(midpoint)
            except Exception as e:
                failed_geoms.append((idx, str(e)))
        
        midpoints = np.array(midpoints)
        
        if len(failed_geoms) > 0:
            print(f"  ⚠️  {len(failed_geoms)} Geometrien fehlgeschlagen:")
            for idx, err in failed_geoms[:5]:  # Nur erste 5 zeigen
                print(f"     - Edge {idx}: {err}")
        
        print(f"  ✅ Mittelpunkte berechnet: {len(midpoints)} Punkte")
        print(f"     X-Range: [{midpoints[:, 0].min():.4f}, {midpoints[:, 0].max():.4f}]")
        print(f"     Y-Range: [{midpoints[:, 1].min():.4f}, {midpoints[:, 1].max():.4f}]")
        
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════════════════════
    # 5. KD-Tree Query
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 5] KD-Tree Query")
    try:
        distances, idx = tree.query(midpoints)
        
        print(f"  ✅ Query erfolgreich")
        print(f"     Min Distanz: {distances.min():.6f} m")
        print(f"     Max Distanz: {distances.max():.6f} m")
        print(f"     Mean Distanz: {distances.mean():.6f} m")
        
        # Warnung bei großen Distanzen
        far_matches = np.sum(distances > 5000)  # > 5 km
        if far_matches > 0:
            print(f"  ⚠️  {far_matches} Matches mit Distanz > 5 km!")
            print(f"     → Edges liegen wahrscheinlich außerhalb des Grids!")
        
        cell_i = i_all[idx]
        cell_j = j_all[idx]
        
        print(f"  ✅ cell_i Range: [{cell_i.min()}, {cell_i.max()}]")
        print(f"  ✅ cell_j Range: [{cell_j.min()}, {cell_j.max()}]")
        
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    # ═══════════════════════════════════════════════════════════════
    # 6. Zurück in Graph schreiben
    # ═══════════════════════════════════════════════════════════════
    print("\n[SCHRITT 6] In Graph schreiben")
    try:
        cell_ids = np.char.add(cell_i.astype(str), "_")
        cell_ids = np.char.add(cell_ids, cell_j.astype(str))
        
        successfully_set = 0
        for (u, v, k), i, j, cid in zip(valid_edges, cell_i, cell_j, cell_ids):
            G.edges[u, v, k]["cell_i"] = int(i)
            G.edges[u, v, k]["cell_j"] = int(j)
            G.edges[u, v, k]["cell_id"] = str(cid)
            successfully_set += 1
        
        print(f"  ✅ Erfolgreich gesetzt: {successfully_set} Edges")
        
        # Zähle wie viele Edges cell_i haben
        edges_with_cellid = sum(1 for u, v, k, d in G.edges(keys=True, data=True) 
                                if "cell_i" in d)
        print(f"  ✅ Gesamt Edges mit cell_i: {edges_with_cellid}")
        
        return G
        
    except Exception as e:
        print(f"  ❌ FEHLER: {e}")
        return None
    
    print("\n" + "="*80 + "\n")


# ═══════════════════════════════════════════════════════════════════════════
# Zusätz-Diagnose: Prüfe Overlap zwischen Graph und Grid
# ═══════════════════════════════════════════════════════════════════════════
def check_spatial_overlap(G, nc_filepath, lat_name="lat", lon_name="lon"):
    """
    Prüft ob der Graph räumlich im Grid-Bereich liegt.
    """
    print("\n" + "="*80)
    print("🗺️  SPATIAL OVERLAP CHECK")
    print("="*80)
    
    # Graph Bounds
    lats = []
    lons = []
    for u, v, k, data in G.edges(keys=True, data=True):
        if "geometry" in data:
            geom = data["geometry"]
            coords = list(geom.coords)
            for lon, lat in coords:
                lons.append(lon)
                lats.append(lat)
    
    if len(lats) == 0:
        print("  ❌ Keine Geometrien im Graph!")
        return
    
    graph_lat_min, graph_lat_max = min(lats), max(lats)
    graph_lon_min, graph_lon_max = min(lons), max(lons)
    
    print(f"\n📍 Graph Bounds:")
    print(f"   Lat: [{graph_lat_min:.6f}, {graph_lat_max:.6f}]")
    print(f"   Lon: [{graph_lon_min:.6f}, {graph_lon_max:.6f}]")
    
    # Grid Bounds
    ds = xr.open_dataset(nc_filepath)
    lat = ds[lat_name].values
    lon = ds[lon_name].values
    
    grid_lat_min, grid_lat_max = lat.min(), lat.max()
    grid_lon_min, grid_lon_max = lon.min(), lon.max()
    
    print(f"\n🎯 Grid Bounds:")
    print(f"   Lat: [{grid_lat_min:.6f}, {grid_lat_max:.6f}]")
    print(f"   Lon: [{grid_lon_min:.6f}, {grid_lon_max:.6f}]")
    
    # Check Overlap
    lat_overlap = not (graph_lat_max < grid_lat_min or graph_lat_min > grid_lat_max)
    lon_overlap = not (graph_lon_max < grid_lon_min or graph_lon_min > grid_lon_max)
    
    print(f"\n📊 Overlap:")
    if lat_overlap and lon_overlap:
        print(f"   ✅ Graph liegt im Grid-Bereich")
    else:
        print(f"   ❌ KEIN OVERLAP!")
        if not lat_overlap:
            print(f"      Lat-Problem: Graph [{graph_lat_min:.4f}, {graph_lat_max:.4f}] "
                  f"vs Grid [{grid_lat_min:.4f}, {grid_lat_max:.4f}]")
        if not lon_overlap:
            print(f"      Lon-Problem: Graph [{graph_lon_min:.4f}, {graph_lon_max:.4f}] "
                  f"vs Grid [{grid_lon_min:.4f}, {grid_lon_max:.4f}]")
    
    ds.close()
    print("\n" + "="*80 + "\n")

In [ ]:
nc_filepath = 'C:/Users/tobia/Documents/FHNW/VPRouting/backend/nc_folder/NC_for_Cellid.nc'
diagnose_get_cellid(G, nc_filepath)

In [ ]:
G_gpdf = ox.convert.graph_to_gdfs(G, nodes=True, edges=True, node_geometry=True, fill_edge_geometry=True)

edges = G_gpdf[1]
nodes = G_gpdf[0]